# Streamgraph mensal de autores
Esta visualização usa somente contribuições com mês conhecido, sem atribuir mês às datas anuais.
O ranking é recalculado pela produção com mês: seleciono os 15% mais produtivos entre os nomes de autores com pelo menos uma contribuição elegível. O resultado pode ter autores diferentes do gráfico anual.
As autorias, nomes e IDs são os preparados no notebook 05. A saída anual permanece disponível.


In [1]:
from pathlib import Path
from collections import Counter
import sys
import os
import math
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import FileLink, display

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root))
from src import config as cfg
from src.autores import is_collective_author
from src.io_utils import atomic_json, atomic_parquet
from src.viz import export_author_streamgraph

cfg.ensure_directories()
pasta = cfg.CURATED / "autores"
autores = pd.read_parquet(pasta / "autores.parquet")


## Selecionar nomes pela produção com mês conhecido
Retiro nomes sem nenhuma contribuição com mês e grupos ou instituições identificados por padrões no campo de autoria. A identificação usa o texto dos metadados e pode deixar casos ambíguos. Uso a contagem `publicacoes_com_mes` do notebook 05 para ordenar o ranking, com desempate pela chave do nome. O corte é o teto de 15% dos elegíveis.
O campo `publicacoes_sem_mes` é preservado apenas para informar quantas contribuições foram excluídas; ele não entra no ranking, nos filtros ou nas séries.


In [2]:
com_mes = autores.loc[autores["publicacoes_com_mes"].gt(0)]
coletivos = is_collective_author(com_mes["autor_chave"])
elegiveis = com_mes.loc[~coletivos].copy()
elegiveis["publicacoes_total"] = elegiveis["publicacoes_com_mes"]
elegiveis = elegiveis.sort_values(["publicacoes_total", "autor_chave"], ascending=[False, True]).reset_index(drop=True)
elegiveis["posicao"] = range(1, len(elegiveis) + 1)
quantidade = math.ceil(cfg.AUTHOR_TOP_FRACTION * len(elegiveis))
elegiveis["selecionado"] = elegiveis["posicao"].le(quantidade)
selecionados = elegiveis.loc[elegiveis["selecionado"]].copy()
chaves = set(selecionados["autor_chave"])
display(selecionados[["posicao", "autor_nome", "publicacoes_total", "publicacoes_sem_mes"]].head(25))


,posicao,autor_nome,publicacoes_total,publicacoes_sem_mes
0,1,"Wang, Wei",408,34
1,2,"Zhang, Wei",343,31
2,3,"Wang, Jing",336,26
3,4,"Liu, Yang",331,19
4,5,"Li, Jing",277,22
5,6,"Mahase, Elisabeth",277,68
6,7,"Li, Yan",272,26
7,8,"Li, Wei",257,27
8,9,"Zhang, Li",257,22
9,10,"Li, Li",256,18


## Contar as contribuições por mês em blocos
Leio apenas a chave do autor e o período mensal das autorias atuais. Registros sem mês são descartados antes da agregação. Acumulo apenas as contagens dos nomes selecionados, sem carregar todas as autorias de uma vez.
Confiro se a soma dos meses corresponde à produção com mês usada no ranking.


In [3]:
contagens = Counter()
for bloco in pq.ParquetFile(cfg.STAGED / "12_autorias.parquet").iter_batches(
        batch_size=50000, columns=["autor_chave", "periodo_mensal"]):
    pares = bloco.to_pandas().dropna(subset=["periodo_mensal"])
    pares = pares.loc[pares["autor_chave"].astype(object).isin(chaves)]
    parte = pares.groupby(["autor_chave", "periodo_mensal"]).size()
    contagens.update(parte.to_dict())
mensal = pd.DataFrame([(nome, periodo, total) for (nome, periodo), total in contagens.items()],
                     columns=["autor_chave", "periodo", "publicacoes"])
del contagens
mensal = mensal.sort_values(["autor_chave", "periodo"]).reset_index(drop=True)
somas = mensal.groupby("autor_chave")["publicacoes"].sum().reindex(selecionados["autor_chave"], fill_value=0)
assert (somas.to_numpy() == selecionados["publicacoes_total"].to_numpy()).all(), "As séries mensais não correspondem ao ranking."
atomic_parquet(pasta / "publicacoes_mensais.parquet", mensal)
selecionados.to_csv(cfg.OUTPUTS / "tables/autores_mensal_top15_porcento.csv", index=False, encoding="utf-8-sig")
selecionados.head(cfg.AUTHOR_DEFAULT_VISIBLE).to_csv(
    cfg.OUTPUTS / "tables/autores_mensal_top25.csv", index=False, encoding="utf-8-sig")


## Registrar o recorte e exportar o HTML mensal
O gráfico cobre os 36 meses de 2020 a 2022. Meses sem contribuições elegíveis recebem zero; isso não indica cobertura completa da base. O release é de junho de 2022.
Filtros, posições e totais consideram apenas contribuições com mês conhecido. O HTML inicia com os 25 primeiros nomes e permite alterar a quantidade exibida. Não há amostragem.


In [4]:
metodologia = {
    "granularidade": "mensal", "fracao": cfg.AUTHOR_TOP_FRACTION,
    "nomes_identificados": len(autores), "nomes_com_mes": len(com_mes),
    "nomes_coletivos_excluidos": int(coletivos.sum()),
    "autorias_coletivas_com_mes_excluidas": int(com_mes.loc[coletivos, "publicacoes_com_mes"].sum()),
    "nomes_elegiveis": len(elegiveis),
    "nomes_selecionados": len(selecionados),
    "autorias_com_mes": int(autores["publicacoes_com_mes"].sum()),
    "autorias_sem_mes_excluidas": int(autores["publicacoes_sem_mes"].sum()),
    "autorias_com_mes_selecionadas": int(selecionados["publicacoes_total"].sum()),
    "ranking": "15% dos nomes elegíveis, excluídos os grupos e instituições identificados; desempate pela chave.",
    "datas": "Somente períodos mensais conhecidos; sem imputação de mês.",
}
atomic_json(pasta / "metodologia_mensal.json", metodologia)
html = export_author_streamgraph(selecionados, mensal, metodologia, monthly=True)
display(pd.Series(metodologia).to_frame("Recorte mensal"))
display(FileLink(os.path.relpath(html, Path.cwd())))


,Recorte mensal
granularidade,mensal
fracao,0.15
nomes_identificados,1784849
nomes_com_mes,1531537
nomes_coletivos_excluidos,1069
autorias_coletivas_com_mes_excluidas,1577
nomes_elegiveis,1530468
nomes_selecionados,229571
autorias_com_mes,2835608
autorias_sem_mes_excluidas,421320


C:\UTFPR\TCC\tcc-data-viz\tcc-data-viz-main\outputs\visualizations\streamgraph_autores_mensal.html